# DriveWise: Metadata-Aware-Automotive-RAG-Assistant

This notebook shows how the DriveWise pipeline works on car brochure text. It only uses the brochure for the selected car, so the answer stays specific to that model.

What it does:
- split brochures into sections
- create embeddings and store them in FAISS
- filter by brand and model before search
- re-rank the search results
- generate an answer from the best match
- log and evaluate the result

The main code is in `src/`, and this notebook just runs it.

## Step 0: Install dependencies

If needed, uncomment the cell below and run it once. The embedding, re-ranking, and generation models are downloaded from HuggingFace the first time they are used.

In [ ]:
# !pip install sentence-transformers faiss-cpu transformers torch pandas streamlit


## Step 1: Load and chunk the brochures

`src/chunker.py` reads each `.txt` file in `data/brochures/` and splits it at `[SECTION: ...]` markers. Each chunk stays on one topic, so the model sees complete information instead of broken fragments.

Each chunk also stores metadata such as brand, model, section, page number, and brochure version. That metadata lets us narrow the search to one car before doing similarity matching.

In [ ]:
import pandas as pd
from src.chunker import load_all_brochures

chunks = load_all_brochures("data/brochures")
df = pd.DataFrame(chunks)
print(f"Loaded {len(df)} chunks from {df['source_file'].nunique()} brochures")
df.head()


## Step 2: Embedding model + FAISS vector database

`src/retriever.py` uses `all-MiniLM-L6-v2` to turn each chunk into a 384-dimensional vector. Similar chunks stay close together even when the wording is different.

The vectors are stored in a FAISS `IndexFlatIP` index for fast similarity search.

In [ ]:
from src.retriever import VectorStore

print("Loading embedding model and building the FAISS index (first run downloads the model)...")
vector_store = VectorStore(chunks)
print(f"Indexed {len(chunks)} chunks into FAISS.")


## Step 3: Metadata filtering + vector search (retrieval)

`vector_store.search()` first filters chunks by brand and model, then embeds the query and finds the most similar chunks within that subset.

That filter matters because otherwise a question about the Hyundai Creta could pull a similar chunk from another brochure.

In [ ]:
results = vector_store.search("What is the mileage?", brand="Hyundai", model="Creta", top_k=5)
for r in results:
    print(f"[{r['chunk']['section']} | page {r['chunk']['page_number']}] score={r['score']:.3f}")
    print(r['chunk']['text'][:100], "...\n")


## Step 4: Re-ranking + context window control

`src/reranker.py` uses the `cross-encoder/ms-marco-MiniLM-L-6-v2` model to re-score the shortlist from Step 3.

The cross-encoder reads the query and chunk together, so it is better at judging whether a chunk actually answers the question. We only run it on a small shortlist because it is slower than the first search step.

After re-ranking, only the best chunk is passed to the LLM.

In [ ]:
from src.reranker import rerank

print("Loading cross-encoder re-ranking model...")
best_chunks = rerank("What is the mileage?", results, top_n=1)
for r in best_chunks:
    print(f"Best match: [{r['chunk']['section']}] rerank_score={r['rerank_score']:.3f}")


## Step 5: Answer generation + source attribution

`src/generator.py` builds a prompt with the best chunk as context and asks `google/flan-t5-small` to answer using only that text. This keeps the answer tied to the brochure instead of the model's training data.

The output includes the answer plus source details: brand, model, section, page number, and source file.

In [ ]:
from src.generator import generate_answer

print("Loading generation model (flan-t5-small)...")
result = generate_answer("What is the mileage?", best_chunks)
print("Answer:", result["answer"])
print("Sources:", result["sources"])


### Calibrating the relevance threshold

`pipeline.ask()` compares the top re-ranker score with `min_rerank_score`. If the score is too low, it returns "I don't have information about that" instead of forcing an answer.

Run the cell below to compare an in-scope and out-of-scope question, then pick a `min_rerank_score` value between the two scores.

In [ ]:
in_scope = rerank("What is the mileage?", vector_store.search("What is the mileage?", brand="Hyundai", model="Creta", top_k=5), top_n=1)
out_of_scope = rerank("What is the price?", vector_store.search("What is the price?", brand="Hyundai", model="Creta", top_k=5), top_n=1)

print("In-scope question rerank_score:", in_scope[0]["rerank_score"] if in_scope else None)
print("Out-of-scope question rerank_score:", out_of_scope[0]["rerank_score"] if out_of_scope else None)
print("Pick a min_rerank_score threshold between these two numbers in src/pipeline.py")


## Step 6: The full pipeline (all steps wired together)

`src/pipeline.py` wraps Steps 1 to 5 in a `DriveWisePipeline` class with a simple `.ask(brand, model, query)` interface. It also applies the threshold check and logs every query.

This is the same class used by `app_web.py`, so the notebook and the web app use the same logic. Adding a new car just means dropping another `.txt` brochure into `data/brochures/`.

In [ ]:
from src.pipeline import DriveWisePipeline

pipeline = DriveWisePipeline()
print("Available cars:", pipeline.list_available_cars())


## Step 7: Logging and monitoring

Every `pipeline.ask()` call writes one JSON line to `logs/query_log.jsonl`. Each entry stores the query, the retrieved chunks, the response time, and whether the request succeeded.

JSONL is a good fit here because new entries can be appended one line at a time and each line can be read on its own.

The cell below runs three demo queries and then reads the log back to confirm the entries were written.

In [ ]:
r1 = pipeline.ask("Hyundai", "Creta", "What is the mileage of the diesel variant?")
print(r1["answer"])
print(r1["sources"])
print()

r2 = pipeline.ask("Tata", "Nexon", "Does it have electronic stability control?")
print(r2["answer"])
print(r2["sources"])
print()

# An out-of-scope question should fail gracefully, not invent an answer
r3 = pipeline.ask("Maruti Suzuki", "Baleno", "What is the price of this car?")
print(r3["answer"])
print(r3["sources"])


In [ ]:
import json
with open("logs/query_log.jsonl") as f:
    for line in f:
        print(json.dumps(json.loads(line), indent=2))


## Step 8: Evaluation (answer correctness, faithfulness, context relevance)

`src/evaluator.py` runs the pipeline on a small test set of five hand-written questions where the expected keyword and correct section are already known. It checks three things:

- Answer correctness: does the generated answer include the expected keyword?
- Faithfulness: does the answer stay within what the retrieved chunk actually says?
- Context relevance: did retrieval return a chunk that matches the question?

These checks give a quick picture of where the pipeline works well and where it needs tuning.

In [ ]:
from src.evaluator import run_evaluation

def ask_fn(brand, model, query):
    return pipeline.ask(brand, model, query)

run_evaluation(ask_fn)


## Conclusion

This notebook shows the DriveWise pipeline end to end: loading brochures, chunking them, retrieving relevant context, generating answers with citations, logging queries, and running a small evaluation.

The main design choices were:
- Filter by metadata before similarity search so answers stay car-specific
- Chunk by section so topics stay intact
- Use a fast bi-encoder first and a cross-encoder second
- Apply a relevance threshold so out-of-scope questions fail gracefully

The same `src/` modules also power the Streamlit app in `app_web.py`, so both interfaces use the same logic.